In [1]:
library(Signac)
library(Seurat)
library(dplyr)
library(GenomeInfoDb)
library(EnsDb.Hsapiens.v86)
library(BSgenome.Hsapiens.UCSC.hg38)
library(ggplot2)
library(patchwork)
library(cowplot)
library(viridis)
library(stringr)
set.seed(1234)
source('/data/work/00.script/colorlist.R')
source('/data/work/00.script/Seurat_helper.r')
source('/data/work/00.script/skin_utiltity.r')

Attaching SeuratObject


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required pack

In [2]:
color = c("#FFFF00", "#1CE6FF", "#FF34FF", "#FF4A46", "#008941",
             "#006FA6", "#A30059", "#FFE4E1", "#0000A6", "#63FFAC",
             "#B79762", "#004D43", "#8FB0FF", "#997D87", "#5A0007",
             "#809693", "#1B4400", "#4FC601", "#3B5DFF", "#FF2F80",
             "#BA0900", "#6B7900", "#00C2A0", "#FFAA92", "#FF90C9",
             "#B903AA", "#DDEFFF", "#7B4F4B", "#A1C299", "#0AA6D8",
             "#00A087FF", "#4DBBD5FF", "#E64B35FF", "#3C5488FF", "#F38400",
             "#A1CAF1", "#C2B280", "#848482", "#E68FAC", "#0067A5",
             "#F99379", "#604E97", "#F6A600", "#B3446C", "#DCD300",
             "#882D17", "#8DB600", "#654522", "#E25822", "#2B3D26",
             "#191970", "#000080",
             "#6495ED", "#1E90FF", "#00BFFF", "#00FFFF", "#FF1493",
             "#FF00FF", "#A020F0", "#63B8FF", "#008B8B", "#54FF9F",
             "#00FF00", "#76EE00", "#FFF68F","#CCCCCC", "#999999",
             "#76EE01", "#FFF66F")

In [6]:
rna = readRDS("/data/users/lijiashan/lijiashan_d8180e14f1f4437c92f3c2850c15bb7e/online/03.Opening/01.Pro_celltype_1120/AA_dmt_81w_1128.rds")

In [7]:
rna

An object of class Seurat 
51352 features across 814128 samples within 1 assay 
Active assay: RNA (51352 features, 3000 variable features)
 5 dimensional reductions calculated: pca, harmony, tsne, umap, dmt

In [5]:
colnames(rna@meta.data)

[1] "orig.ident"               "nCount_RNA"              
 [3] "nFeature_RNA"             "sampleid"                
 [5] "percent.mt"               "scDblFinder.class"       
 [7] "scDblFinder.score"        "batch"                   
 [9] "sample"                   "Acute"                   
[11] "SALT"                     "Type"                    
[13] "Sex"                      "Grade"                   
[15] "cellid"                   "group"                   
[17] "nCount_RNA_Deviation"     "nFeature_RNA_Deviation"  
[19] "percent.mt_RNA_Deviation" "RNA_snn_res.0.2"         
[21] "RNA_snn_res.0.5"          "RNA_snn_res.0.8"         
[23] "seurat_clusters"          "sub.cluster"             
[25] "celltype"                 "Categories"              
[27] "dmt_leiden"               "dmt_leiden_sub"          
[29] "celltype_dmt_0929"        "Cell_Subtype"            
[31] "celltype_dmt_1029"        "celltype_dmt_1121"       
[33] "Acute1124"

In [6]:
unique(rna$celltype_dmt_1121)

[1] "IFEB"      "MAST"      "IRSC"      "SMC"       "VENDO"     "MCC"      
 [7] "DFB"       "HFSC_pro"  "CENDO"     "CD4Tm"     "EGCC2"     "EGCC1"    
[13] "EGD"       "B"         "ILC"       "IFES"      "ORSS"      "PCT"      
[19] "Outer_II"  "MELA"      "HFSC"      "MACRO"     "AENDO"     "CD8E"     
[25] "IRS"       "ORSB"      "LGS"       "IFE_pro"   "DS"        "LENDO"    
[31] "DP"        "MEC"       "SW"        "T"         "Inner_ii"  "CD4T_un"  
[37] "CD8_Naive" "IFEG"      "TAC"       "CD4T_pro"  "ORS_pro"   "Ecs_pro"  
[43] "CD8Tex"    "DS_pro"    "NC"

In [10]:
Idents(rna)='celltype_dmt_1121'
rna1= subset(rna, downsample = 500)

In [11]:
rna1

An object of class Seurat 
51352 features across 21569 samples within 1 assay 
Active assay: RNA (51352 features, 3000 variable features)
 5 dimensional reductions calculated: pca, harmony, tsne, umap, dmt

In [19]:
#如果sample有样本细胞数不够就换组别
rna1.list <- SplitObject(rna1, split.by = "Grade")

In [20]:
# 1. 检查每个样本的细胞数
cell_counts <- sapply(rna1.list, ncol)
print(cell_counts)

# 2. 检查每个样本的基因数（高变基因后）
gene_counts <- sapply(rna1.list, function(x) {
  length(VariableFeatures(x))
})
print(gene_counts)

# 3. 找出问题样本
problem_samples <- which(cell_counts < 30 | gene_counts < 30)
print(paste("问题样本索引:", problem_samples))

 Nor   S3   S2   S1   S4 
4469 4639 3538 5544 3379 
 Nor   S3   S2   S1   S4 
3000 3000 3000 3000 3000 
[1] "问题样本索引: "


In [21]:
for (i in 1
:length(rna1.list)) {
    rna1.list[[i]] <- NormalizeData(rna1.list[[i]], verbose = FALSE)
    rna1.list[[i]] <- FindVariableFeatures(rna1.list[[i]], selection.method = "vst", 
        nfeatures = 2000, verbose = FALSE)%>%ScaleData(.,verbose = F) %>% RunPCA(., npcs = 30)
}

PC_ 1 
Positive:  VIM, SPARC, COL3A1, COL6A2, COL6A1, COL1A2, AEBP1, IFITM2, C1R, IFITM3 
	   MGP, TIMP1, COL1A1, SERPING1, DCN, TIMP3, MXRA8, LUM, PRRX1, IGFBP7 
	   PCOLCE, COL6A3, SERPINF1, FSTL1, C1S, THY1, IGFBP4, COL12A1, FBLN2, MFAP4 
Negative:  SFN, DSG1, SERPINB5, LYPD3, DMKN, CALML3, FABP5, OVOL1, SBSN, GJB2 
	   SPINK5, KRT10, KRTDAP, TMEM45A, DSC1, GATA3, METAP2, DAPL1, ZNF185, NRARP 
	   IVL, KRT85, CALML5, DNASE1L2, PPP2R1B, CSTA, POF1B, KRT35, S100A2, MAL2 
PC_ 2 
Positive:  CXCL14, PALLD, COL6A1, COL1A2, DCN, COL6A2, COL1A1, KRT14, COL3A1, DMKN 
	   KRT5, DCD, CALD1, MGP, DSG1, COL12A1, AEBP1, CALML3, KRT10, LUM 
	   C1R, SFRP1, KRT17, CTSK, MXRA5, COL5A2, MXRA8, PRRX1, FBLN1, LINC00511 
Negative:  TYROBP, HLA-DPB1, HLA-DRA, LCP1, HLA-DQA1, FCER1A, HLA-DQB1, HLA-DRB1, RGS1, SRGN 
	   HLA-DPA1, AIF1, LST1, LAPTM5, GPR183, CD83, HLA-DQA2, HLA-DQB2, HLA-DMB, HLA-DMA 
	   PLEK, PTPRC, CPVL, CD74, HLA-DRB5, CD86, ITGB2, FCER1G, MNDA, CD1A 
PC_ 3 
Positive:  DCD, PECAM1, SOX5

In [22]:
features <-SelectIntegrationFeatures(object.list = rna1.list)
rna1.anchors <- FindIntegrationAnchors(object.list = rna1.list, normalization.method = "LogNormalize",anchor.features = features,
                                      reduction = "rpca",  dims = 1:30)
rna1<- IntegrateData(anchorset = rna1.anchors, dims = 1:30)

rna1 <- ScaleData(rna1, verbose = FALSE)
rna1 <- RunPCA(rna1, npcs = 30, verbose = FALSE)
rna1 <- RunUMAP(rna1, reduction = "pca", dims = 1:30)

Scaling features for provided objects

Computing within dataset neighborhoods

Finding all pairwise anchors

Projecting new data onto SVD

Projecting new data onto SVD

Finding neighborhoods

Finding anchors

	Found 2875 anchors

Projecting new data onto SVD

Projecting new data onto SVD

Finding neighborhoods

Finding anchors

	Found 2323 anchors

Projecting new data onto SVD

Projecting new data onto SVD

Finding neighborhoods

Finding anchors

	Found 3687 anchors

Projecting new data onto SVD

Projecting new data onto SVD

Finding neighborhoods

Finding anchors

	Found 2762 anchors

Projecting new data onto SVD

Projecting new data onto SVD

Finding neighborhoods

Finding anchors

	Found 4017 anchors

Projecting new data onto SVD

Projecting new data onto SVD

Finding neighborhoods

Finding anchors

	Found 3800 anchors

Projecting new data onto SVD

Projecting new data onto SVD

Finding neighborhoods

Finding anchors

	Found 1608 anchors

Projecting new data onto SVD

Projecting new

In [23]:
rna1

An object of class Seurat 
53352 features across 21569 samples within 2 assays 
Active assay: integrated (2000 features, 2000 variable features)
 1 other assay present: RNA
 2 dimensional reductions calculated: pca, umap

In [ ]:
DefaultAssay(rna1) <- 'RNA'
rna1 = rna1 %>% NormalizeData() %>% FindVariableFeatures() %>% ScaleData()

In [ ]:
rna = rna1

In [ ]:
DefaultAssay(rna) <- 'RNA'
rna$Tech = 'RNA'

In [ ]:
rna

In [3]:
atac = readRDS("/data/work/02.ATAC/02.new_1203/AA_ATAC_rlsi_anno_filter_1211.rds")
DefaultAssay(atac) <- "RNA"
atac$Tech = 'ATAC'

In [4]:
atac

An object of class Seurat 
205046 features across 21285 samples within 2 assays 
Active assay: RNA (19607 features, 0 variable features)
 1 other assay present: ATAC
 2 dimensional reductions calculated: integrated_lsi, umap

In [ ]:
##### 03. CCA integrated #####
DefaultAssay(atac) <- "RNA"
features <- SelectIntegrationFeatures(object.list = list(rna, atac))
anchors <- FindIntegrationAnchors(
  object.list = list(rna, atac),
  anchor.features = features,
  assay = c('RNA', 'RNA')
)

In [ ]:
# integrate data and create a new merged object
integrated <- IntegrateData(
  anchorset = anchors,
  weight.reduction = c('pca', 'integrated_lsi'),
  dims = 2:30,
  preserve.order = TRUE
)

In [ ]:
rm(rna,atac)
gc()

In [ ]:
integrated = ScaleData(integrated)
integrated <- RunPCA(
  object = integrated,
  npcs = 30,
  reduction.name = 'integr_PCA'
)

In [ ]:
integrated <- RunUMAP(
  object = integrated,
  dims = 1:30,
  reduction = 'integr_PCA',
  min.dist = 0.3
)

In [ ]:
saveRDS(integrated, "/data/work/02.ATAC/02.new_1203/AA_RNA_ATAC_inter_CCA_1214.rds")

Idents(integrated) = 'Tech'
p <- DimPlot(integrated, group.by = 'Tech', pt.size = 0.001, shuffle = T, raster = F, cols = best_color) + ggplot2::ggtitle("HF_RNA&ATAC_Integrated")
ggsave(p, file = "/data/work/02.ATAC/02.new_1203/AA_RNA_ATAC_Integrated_CCA.png",width = 7, height = 5)

# p = DimPlot(integrated, group.by = c("celltype","predicted.id"), pt.size = 0.001, raster = F, cols = colsblack)
# ggsave(p, file = 'HF_RNA_ATAC_pair_inter_CCA_celltype.png', width = 27, height = 10)
# 
# integrated$Celltype_Predict = integrated$celltype
# integrated@meta.data[integrated$Tech == 'ATAC', ]$Celltype_Predict = integrated@meta.data[integrated$Tech == 'ATAC', ]$predicted.id
# 
# p = DimPlot(integrated, group.by = 'Celltype_Predict', pt.size = 0.001, raster = F) + ggplot2::ggtitle("HF RNA (celltype) & ATAC (predicted.id)")
# ggsave(p, file = "HF_RNA_ATAC_pair_Integrated_CCA_celltype.png",width = 18, height = 6)

print('AA integrated (CCA) has finished!')